# 딥러닝 기온 예측
## ASOS 서울(STN 108) 과거 3일 시간별 기온 → 다음날 시간별 기온 (24개)

| 항목 | 내용 |
|------|------|
| **데이터** | ASOS 서울(STN 108) 시간별 기온 (2013–2022) |
| **입력** | 과거 3일 시간별 기온 (72 timesteps) |
| **출력** | 다음날 시간별 기온 (24 values, 0시~23시) |
| **Train** | 2013–2020 |
| **Val** | 2021 |
| **Test** | 2022 |
| **모델** | MLP · LSTM · Transformer |

---
## Step 1. 패키지 임포트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/weather-climate-ai-tutorials

In [ ]:
# 나눔 고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pathlib, matplotlib, matplotlib.font_manager as fm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── 한글 폰트 설치 및 등록 ───────────────────────────────────────────
font_path = pathlib.Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'NanumGothic.ttf'

if not font_path.exists():
    print('NanumGothic 폰트 다운로드 중...')
    try:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path
        )
        print('다운로드 완료')
    except Exception as e:
        print(f'다운로드 실패: {e}')

if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    _font_name = fm.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams['font.family'] = _font_name
    print(f'폰트 등록 완료: {_font_name}')
else:
    for sys_path in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf']:
        if pathlib.Path(sys_path).exists():
            fm.fontManager.addfont(sys_path)
            plt.rcParams['font.family'] = fm.FontProperties(fname=sys_path).get_name()
            print(f'시스템 폰트 사용: {sys_path}')
            break
    else:
        print('한글 폰트 없음 → 기본 폰트 사용')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')

---
## Step 2. 설정

In [ ]:
ASOS_DIR   = './asos_data'
TRAIN_YEARS = list(range(2013, 2021))   # 2013–2020
VAL_YEARS   = [2021]
TEST_YEARS  = [2022]

SEQ_LEN    = 72      # 3일 × 24시간
N_FEATURES = 1       # 기온만 사용
N_TARGETS  = 24      # 다음날 시간별 기온 (0시~23시)

BATCH_SIZE = 64
EPOCHS     = 100
LR         = 1e-3

print(f'입력 시퀀스: {SEQ_LEN}시간 (3일)')
print(f'출력: {N_TARGETS}시간 (다음날 0시~23시)')
print(f'Train: {TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]}, Val: {VAL_YEARS}, Test: {TEST_YEARS}')

---
## Step 3. 데이터 로드 및 전처리

In [ ]:
def load_asos_hourly(years):
    """지정 연도 merged CSV 로드 → 시간별 기온 Series 반환 (KST)."""
    dfs = []
    for yr in years:
        path = f'{ASOS_DIR}/asos_{yr}_merged.csv'
        df = pd.read_csv(path, dtype=str, low_memory=False)
        df['datetime'] = pd.to_datetime(df['TM'], format='%Y%m%d%H%M', errors='coerce')
        df['TA'] = pd.to_numeric(df['TA'], errors='coerce')
        df.loc[df['TA'] <= -9, 'TA'] = float('nan')
        df = df.dropna(subset=['datetime', 'TA'])
        dfs.append(df[['datetime', 'TA']])

    df_all = pd.concat(dfs, ignore_index=True)
    df_all = df_all.set_index('datetime').sort_index()
    # 1시간 간격으로 리인덱싱 (결측은 선형 보간)
    full_idx = pd.date_range(df_all.index.min(), df_all.index.max(), freq='1h')
    ta = df_all['TA'].reindex(full_idx).interpolate('linear', limit=3)
    return ta

print('데이터 로드 중...')
ta_train = load_asos_hourly(TRAIN_YEARS)
ta_val   = load_asos_hourly(VAL_YEARS)
ta_test  = load_asos_hourly(TEST_YEARS)

print(f'Train: {ta_train.shape[0]:,}시간  ({ta_train.index[0].date()} ~ {ta_train.index[-1].date()})')
print(f'Val  : {ta_val.shape[0]:,}시간  ({ta_val.index[0].date()} ~ {ta_val.index[-1].date()})')
print(f'Test : {ta_test.shape[0]:,}시간  ({ta_test.index[0].date()} ~ {ta_test.index[-1].date()})')

---
## Step 4. 시퀀스 생성

In [ ]:
def make_sequences(ta_series):
    """
    X : (N, SEQ_LEN, 1)  — 과거 72시간 기온
    y : (N, 24)          — 다음날 시간별 기온 (0시~23시)
    """
    X_list, y_list, y_dates = [], [], []

    # 날짜 목록 (데이터가 있는 날만)
    daily_dates = ta_series.resample('1D').mean().dropna().index

    for target_date in daily_dates:
        # 다음날 24시간 기온 (0시~23시)
        next_start = target_date
        next_end   = target_date + pd.Timedelta(hours=23)
        next_chunk = ta_series.loc[next_start:next_end]
        if len(next_chunk) != 24 or next_chunk.isnull().any():
            continue

        # 입력: target_date 기준 72시간 전 ~ 1시간 전
        end   = target_date - pd.Timedelta(hours=1)
        start = end - pd.Timedelta(hours=SEQ_LEN - 1)
        chunk = ta_series.loc[start:end]
        if len(chunk) != SEQ_LEN or chunk.isnull().any():
            continue

        X_list.append(chunk.values.reshape(SEQ_LEN, 1))
        y_list.append(next_chunk.values)          # shape (24,)
        y_dates.append(target_date)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y, y_dates

X_train, y_train, dates_train = make_sequences(ta_train)
X_val,   y_val,   dates_val   = make_sequences(ta_val)
X_test,  y_test,  dates_test  = make_sequences(ta_test)

print(f'Train  X:{X_train.shape}  y:{y_train.shape}')
print(f'Val    X:{X_val.shape}    y:{y_val.shape}')
print(f'Test   X:{X_test.shape}   y:{y_test.shape}')

In [ ]:
# ── StandardScaler (train 데이터로 fit) ───────────────────────────────
scaler_X = StandardScaler()
scaler_y = StandardScaler()

N, T, F = X_train.shape
scaler_X.fit(X_train.reshape(-1, F))
scaler_y.fit(y_train)                  # y_train: (N, 24) → 시간별로 독립 스케일링

def norm_X(X):
    n = X.shape[0]
    return scaler_X.transform(X.reshape(-1, F)).reshape(n, T, F).astype(np.float32)

X_tr = norm_X(X_train);  y_tr = scaler_y.transform(y_train).astype(np.float32)
X_vl = norm_X(X_val);    y_vl = scaler_y.transform(y_val).astype(np.float32)
X_te = norm_X(X_test);   y_te = scaler_y.transform(y_test).astype(np.float32)

print('정규화 완료')
print(f'Train 기온 범위: {y_train.min():.1f} ~ {y_train.max():.1f} °C')

---
## Step 5. Dataset & DataLoader

In [ ]:
class TempDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(TempDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TempDataset(X_vl, y_vl), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TempDataset(X_te, y_te), batch_size=BATCH_SIZE)

print(f'Train {len(train_loader.dataset)}개 / Val {len(val_loader.dataset)}개 / Test {len(test_loader.dataset)}개')

---
## Step 6. 모델 정의

In [ ]:
# ── MLP ──────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                          # (B, 72, 1) → (B, 72)
            nn.Linear(SEQ_LEN, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),     nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),      nn.ReLU(),
            nn.Linear(64, N_TARGETS),
        )
    def forward(self, x): return self.net(x)


# ── LSTM ─────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, hidden=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(N_FEATURES, hidden, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden, N_TARGETS)

    def forward(self, x):                          # x: (B, 72, 1)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])              # 마지막 타임스텝


# ── Transformer ──────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerModel(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(N_FEATURES, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=256,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers)
        self.fc = nn.Linear(d_model, N_TARGETS)

    def forward(self, x):                          # x: (B, 72, 1)
        x = self.pos_enc(self.input_proj(x))       # (B, 72, d_model)
        x = self.transformer(x).mean(dim=1)        # global avg pool
        return self.fc(x)


for name, m in [('MLP', MLP()), ('LSTM', LSTMModel()), ('Transformer', TransformerModel())]:
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<15} params: {n:,}')

---
## Step 7. 학습

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5
    )
    criterion = nn.MSELoss()
    tr_losses, vl_losses = [], []

    for epoch in range(1, epochs + 1):
        # train
        model.train()
        tr = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tr += loss.item() * len(xb)
        tr /= len(train_loader.dataset)

        # val
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl += criterion(model(xb), yb).item() * len(xb)
        vl /= len(val_loader.dataset)

        scheduler.step(vl)
        tr_losses.append(tr)
        vl_losses.append(vl)

        if epoch % 10 == 0:
            print(f'  [{epoch:3d}/{epochs}] train={tr:.4f}  val={vl:.4f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

    return model, tr_losses, vl_losses

In [ ]:
MODELS = {
    'MLP'        : MLP(),
    'LSTM'       : LSTMModel(),
    'Transformer': TransformerModel(),
}

results = {}
for name, model in MODELS.items():
    print(f'\n{"="*50}\n  {name} 학습\n{"="*50}')
    trained, tr_l, vl_l = train_model(model, train_loader, val_loader)
    results[name] = {'model': trained, 'tr_losses': tr_l, 'vl_losses': vl_l}
    print(f'  완료  (best val loss: {min(vl_l):.4f})')

---
## Step 8. 평가 및 시각화

In [ ]:
# ── 학습 곡선 ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    ax.plot(res['tr_losses'], lw=1.5, label='Train')
    ax.plot(res['vl_losses'], lw=1.5, label='Val')
    ax.set_title(f'{name}', fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('학습/검증 Loss 곡선', fontsize=14)
plt.tight_layout()
plt.savefig('temp_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 테스트셋 성능 평가 ────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            p = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(p); trues.append(yb.numpy())
    preds = scaler_y.inverse_transform(np.concatenate(preds))   # (N, 24)
    trues = scaler_y.inverse_transform(np.concatenate(trues))   # (N, 24)
    return preds, trues   # 2D 배열 반환

print(f"{'모델':<14} {'RMSE (°C)':>10} {'MAE (°C)':>10} {'MBE (°C)':>10}")
print('-' * 46)
for name, res in results.items():
    p, t = evaluate(res['model'], test_loader)
    res['preds'] = p; res['trues'] = t
    p_flat, t_flat = p.ravel(), t.ravel()
    rmse = np.sqrt(mean_squared_error(t_flat, p_flat))
    mae  = mean_absolute_error(t_flat, p_flat)
    mbe  = (p_flat - t_flat).mean()
    print(f'{name:<14} {rmse:>10.3f} {mae:>10.3f} {mbe:>10.3f}')

In [ ]:
# ── 2022년 시간별 예측 시계열 ─────────────────────────────────────────
model_colors = {'MLP': '#E63946', 'LSTM': '#2196F3', 'Transformer': '#4CAF50'}
dates_arr = np.array(dates_test)

# (N, 24) → 연속 시간별 시계열로 펼치기
def flatten_hourly(preds_2d, dates):
    hours, vals = [], []
    for i, d in enumerate(dates):
        for h in range(24):
            hours.append(pd.Timestamp(d) + pd.Timedelta(hours=h))
            vals.append(preds_2d[i, h])
    return hours, np.array(vals)

trues_orig = list(results.values())[0]['trues']
hour_dates, true_vals = flatten_hourly(trues_orig, dates_arr)

fig, ax = plt.subplots(figsize=(18, 5))
ax.plot(hour_dates, true_vals, color='black', lw=1.2, alpha=0.85, label='ASOS 실제', zorder=5)
for name, res in results.items():
    _, pred_vals = flatten_hourly(res['preds'], dates_arr)
    ax.plot(hour_dates, pred_vals, color=model_colors[name], lw=0.8, alpha=0.7, label=name)
ax.set_xlabel('날짜 (2022년)'); ax.set_ylabel('기온 (°C)')
ax.set_title('2022년 테스트셋 — 시간별 기온 예측 vs 실제', fontsize=13)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('temp_forecast_2022_hourly.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 시간대별 평균 RMSE (0시~23시) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
hours_x = np.arange(24)
for name, res in results.items():
    hourly_rmse = [
        np.sqrt(mean_squared_error(res['trues'][:, h], res['preds'][:, h]))
        for h in range(24)
    ]
    ax.plot(hours_x, hourly_rmse, marker='o', ms=4, color=model_colors[name], label=name)
ax.set_xticks(hours_x); ax.set_xlabel('시간 (시)'); ax.set_ylabel('RMSE (°C)')
ax.set_title('시간대별 RMSE (2022년 테스트셋)', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('temp_hourly_rmse_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 산점도 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, res) in zip(axes, results.items()):
    t = res['trues'].ravel()
    p = res['preds'].ravel()
    rmse = np.sqrt(mean_squared_error(t, p))
    mae  = mean_absolute_error(t, p)
    ax.scatter(t, p, s=4, alpha=0.2, color=model_colors[name])
    lim = [t.min() - 1, t.max() + 1]
    ax.plot(lim, lim, 'k--', lw=1.2)
    ax.set_xlabel('실제 기온 (°C)'); ax.set_ylabel('예측 기온 (°C)')
    ax.set_title(f'{name}\nRMSE={rmse:.2f}°C  MAE={mae:.2f}°C', fontsize=11)
    ax.grid(alpha=0.3)
plt.suptitle('2022년 시간별 예측 산점도', fontsize=13)
plt.tight_layout()
plt.savefig('temp_scatter_2022_hourly.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 월별 RMSE 비교 ────────────────────────────────────────────────────
dates_pd = pd.DatetimeIndex(dates_test)
months   = dates_pd.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(12)
w = 0.25
for i, (name, res) in enumerate(results.items()):
    monthly_rmse = [
        np.sqrt(mean_squared_error(
            res['trues'][months == m].ravel(),
            res['preds'][months == m].ravel()
        )) if (months == m).any() else 0
        for m in range(1, 13)
    ]
    ax.bar(x + i * w, monthly_rmse, width=w, label=name,
           color=model_colors[name], alpha=0.85)

ax.set_xticks(x + w); ax.set_xticklabels(month_names)
ax.set_xlabel('월'); ax.set_ylabel('RMSE (°C)')
ax.set_title('2022년 월별 RMSE 비교 (시간별 예측)', fontsize=13)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('temp_monthly_rmse_2022_hourly.png', dpi=150, bbox_inches='tight')
plt.show()